In [0]:
#Libraries management
from pyspark import pipelines as pl
from pyspark.sql.functions import *
from pyspark.sql.types import *

volume_path="/Volumes/workspace/damg7370/datastore/Schema_Drift/customer_*.json"

In [0]:
#bronze layer table: cust_bronze_sd
pl.create_streaming_table("demo_cust_bronze_sd_add")

# Ingest the raw data into the bronze table using append flow
@pl.append_flow(
  target = "demo_cust_bronze_sd_add", #object name
  name = "demo_cust_bronze_sd_ingest_add_flow" #flow name
)
def demo_cust_bronze_sd_ingest_add_flow():
  df = (
      spark.readStream
          .format("cloudFiles")
          .option("cloudFiles.format", "json")
          .option("cloudFiles.inferColumnTypes", "true") #auto scan schema 
          #.option("cloudFiles.schemaEvolutionMode", "failOnNewColumns") # schema customer_data_1.json is different than customer_data_2.json so it fails with  [UNKNOWN_FIELD_EXCEPTION.NEW_FIELDS_IN_RECORD_WITH_FILE_PATH] excetion and stops processing
          .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
          .load(f"{volume_path}")
  )
  return df.withColumn("ingestion_datetime", current_timestamp())\
           .withColumn("source_filename", col("_metadata.file_path"))

In [0]:
pl.create_streaming_table(
    name="demo_cust_silver_sd_add",
    expect_all_or_drop={
        "valid_id": "CustomerID IS NOT NULL"
    }
)

@pl.append_flow(
    target="demo_cust_silver_sd_add",
    name="demo_cust_silver_sd_clean_add_flow"
)
def demo_cust_silver_sd_clean_add_flow():
    df = spark.readStream.table("demo_cust_bronze_sd_add")

    # Only changing data types
    df = df.withColumn("SignupDate", col("SignupDate").cast(DateType()))

    return df